In [1]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

import sys
sys.path.append("/Users/admin/eeg-ds004504/")


In [2]:
from config_handler import initiate_config, load_config

initiate_config()

{'data_path': '/Users/admin/eeg-ds004504',
 'derivatives': True,
 'freqBands': {'Alpha': [8, 12],
  'Beta': [12, 30],
  'Delta': [0.5, 4],
  'Theta': [4, 8],
  'custom1': [9, 11]},
 'method': 'welch',
 'stepSize': 0.3,
 'windowLength': 3}

In [3]:
print(load_config())

{'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [5]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

In [6]:
# creating the ultimate most optimised spark session ever muwahaha
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx12g -Xms4g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "256") \

    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)

    # ----------------------------------------
    # Additional advanced options (optional):
    # ----------------------------------------

    # Use Kryo serializer instead of default Java serializer for better performance
    # .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \

    # Increase broadcast join timeout (in seconds) for large models or lookup tables
    # .config("spark.sql.broadcastTimeout", "600") \

    # Fraction of JVM memory reserved for execution and storage (default is 0.6)
    # .config("spark.memory.fraction", "0.8") \

    # Portion of memory reserved for caching/storage (default is 0.5 of memory.fraction)
    # .config("spark.memory.storageFraction", "0.3") \

    # Enable Apache Arrow for efficient pandas-to-Spark conversion (useful with UDFs)
    # .config("spark.sql.execution.arrow.pyspark.enabled", "true") \

    # Finalize and create the Spark session


# spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

Picked up _JAVA_OPTIONS: -Xmx12g -Xms4g
Picked up _JAVA_OPTIONS: -Xmx12g -Xms4g
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/25 10:52:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/25 10:52:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/04/25 10:52:13 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/04/25 10:52:13 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/04/25 10:52:13 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/04/25 10:52:13 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


New Spark session created successfully


In [7]:
def reload_my_modules():
    import importlib
    import populate_schemas
    import feature_extraction
    import schema_definition
    importlib.reload(populate_schemas)
    importlib.reload(feature_extraction)
    importlib.reload(schema_definition)

reload_my_modules()


from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try: 
    # oh btw spark is werid about not finding the config but it always finds it somehow not sure how that works not going to look rn tbh
    # ^ so feature_extraction has extra print statements
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "config_handler.py"))
    print("Added config_handler.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Added feature_extraction.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context
Added config_handler

In [8]:
import pandas as pd

# alz_df_pandas = pd.read_pickle("alz_df_apr10_1355.pkl")
# cntrl_df_pandas = pd.read_pickle("cntrl_df_apr10_1355.pkl")


# alz_df_pandas = pd.read_pickle("features_alz_extra_features_Apr15_1033.pkl")
# cntrl_df_pandas = pd.read_pickle("features_cntrl_extra_features_Apr15_1033.pkl")

alz_df_pandas = pd.read_pickle("features_alz_extra_features_Apr19_2141.pkl")
cntrl_df_pandas = pd.read_pickle("features_cntrl_extra_features_Apr19_2141.pkl")


In [9]:
%%time
alz_df_spark   = spark.createDataFrame(alz_df_pandas)
cntrl_df_spark = spark.createDataFrame(cntrl_df_pandas)

CPU times: user 6min 42s, sys: 4.09 s, total: 6min 47s
Wall time: 6min 50s


In [8]:
alz_df_spark = spark.read.parquet("features_alz_extra_features_Apr19_2141.parquet")
cntrl_df_spark = spark.read.parquet("features_cntrl_extra_features_Apr19_2141.parquet")

In [9]:
#just renaming things now that we understand the types and where things are coming from
alz_df = alz_df_spark
cntrl_df = cntrl_df_spark

In [10]:
alz_df.show()

+---------+-------+---------+--------+-----------+-------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName| FeatureValue|table_type|
+---------+-------+---------+--------+-----------+-------------+----------+
|  sub-008|   ep-0|      Fp1|   Alpha|      Power|  7.020328E-4|      band|
|  sub-008|   ep-0|      Fp1|    Beta|      Power|  3.399499E-4|      band|
|  sub-008|   ep-0|      Fp1|   Delta|      Power|  0.087231696|      band|
|  sub-008|   ep-0|      Fp1|   Theta|      Power| 0.0011391409|      band|
|  sub-008|   ep-0|      Fp1| custom1|      Power| 4.2795436E-4|      band|
|  sub-008|   ep-0|      Fp1|    NULL|TotalEnergy|   0.34805238| electrode|
|  sub-008|   ep-0|      Fp1|    NULL| TotalPower|  0.011235955| electrode|
|  sub-008|   ep-0|      Fp2|   Alpha|      Power| 0.0014689578|      band|
|  sub-008|   ep-0|      Fp2|    Beta|      Power|  5.043481E-4|      band|
|  sub-008|   ep-0|      Fp2|   Delta|      Power|   0.08462298|      band|
|  sub-008| 

# Raw Data Visualization

# Start of data processing

In [11]:
# give each its respective labels
from pyspark.sql.functions import lit
alz_df = alz_df.withColumn("label", lit(1)).repartition(16).persist()
cntrl_df = cntrl_df.withColumn("label", lit(0)).repartition(16).persist()

In [12]:
# union everything
full_df = alz_df.unionByName(cntrl_df)

In [13]:
# Split based on feature type
from pyspark.sql.functions import col

band_df = full_df.filter(col("table_type") == "band")
channel_df = full_df.filter(col("table_type") == "electrode")
epoch_df = full_df.filter(col("table_type") == "epoch")


In [14]:
from pyspark.sql.functions import concat_ws

# Band-level: Electrode_WaveBand_Feature
band_df = band_df.withColumn("pivot", concat_ws("_", "Electrode", "WaveBand", "FeatureName")).repartition(16).persist()

# Channel-level: Electrode_Feature
channel_df = channel_df.withColumn("pivot", concat_ws("_", "Electrode", "FeatureName")).repartition(16).persist()

# Epoch-level: just FeatureName
epoch_df = epoch_df.withColumn("pivot", col("FeatureName")).repartition(16).persist()


In [15]:
from pyspark.sql.functions import first

# Pivot band-level features
band_pivot = band_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot channel-level features
channel_pivot = channel_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot epoch-level features
epoch_pivot = epoch_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

In [16]:
from functools import reduce

# full_df = reduce(
#     lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
#     [band_pivot, channel_pivot, epoch_pivot]
# ).fillna(0.0)


full_df = reduce(
    lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
     [band_pivot, channel_pivot, epoch_pivot]# band_pivot, channel_pivot, epoch_pivot]
).fillna(0.0)
full_df.repartition(16).persist()


25/04/25 10:57:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[SubjectID: string, EpochID: string, label: int, C3_Alpha_Power: float, C3_Beta_Power: float, C3_Delta_Power: float, C3_Theta_Power: float, C3_custom1_Power: float, C4_Alpha_Power: float, C4_Beta_Power: float, C4_Delta_Power: float, C4_Theta_Power: float, C4_custom1_Power: float, Cz_Alpha_Power: float, Cz_Beta_Power: float, Cz_Delta_Power: float, Cz_Theta_Power: float, Cz_custom1_Power: float, F3_Alpha_Power: float, F3_Beta_Power: float, F3_Delta_Power: float, F3_Theta_Power: float, F3_custom1_Power: float, F4_Alpha_Power: float, F4_Beta_Power: float, F4_Delta_Power: float, F4_Theta_Power: float, F4_custom1_Power: float, F7_Alpha_Power: float, F7_Beta_Power: float, F7_Delta_Power: float, F7_Theta_Power: float, F7_custom1_Power: float, F8_Alpha_Power: float, F8_Beta_Power: float, F8_Delta_Power: float, F8_Theta_Power: float, F8_custom1_Power: float, Fp1_Alpha_Power: float, Fp1_Beta_Power: float, Fp1_Delta_Power: float, Fp1_Theta_Power: float, Fp1_custom1_Power: float, Fp2_Alpha

In [17]:
type(full_df)

pyspark.sql.dataframe.DataFrame

In [47]:
full_df.repartition(16).persist()


25/04/25 11:13:44 WARN CacheManager: Asked to cache already cached data.


DataFrame[SubjectID: string, EpochID: string, label: int, C3_Alpha_Power: float, C3_Beta_Power: float, C3_Delta_Power: float, C3_Theta_Power: float, C3_custom1_Power: float, C4_Alpha_Power: float, C4_Beta_Power: float, C4_Delta_Power: float, C4_Theta_Power: float, C4_custom1_Power: float, Cz_Alpha_Power: float, Cz_Beta_Power: float, Cz_Delta_Power: float, Cz_Theta_Power: float, Cz_custom1_Power: float, F3_Alpha_Power: float, F3_Beta_Power: float, F3_Delta_Power: float, F3_Theta_Power: float, F3_custom1_Power: float, F4_Alpha_Power: float, F4_Beta_Power: float, F4_Delta_Power: float, F4_Theta_Power: float, F4_custom1_Power: float, F7_Alpha_Power: float, F7_Beta_Power: float, F7_Delta_Power: float, F7_Theta_Power: float, F7_custom1_Power: float, F8_Alpha_Power: float, F8_Beta_Power: float, F8_Delta_Power: float, F8_Theta_Power: float, F8_custom1_Power: float, Fp1_Alpha_Power: float, Fp1_Beta_Power: float, Fp1_Delta_Power: float, Fp1_Theta_Power: float, Fp1_custom1_Power: float, Fp2_Alpha

# NEED TO MIN MAX AFTER PCA!, should do something line z-score , pca , then min max (optional)

In [51]:
# import dimensionality_reduction
import importlib
try:
    importlib.reload(dimensionality_reduction)
except:
    pass
from dimensionality_reduction import min_max_normalize, normalize_by_column, normalize_by_column_per_subject_wide
feature_cols = [c for c in full_df.columns if c not in ("SubjectID", "EpochID", "label")]


full_df = normalize_by_column_per_subject_wide(full_df, feature_cols)



full_df.repartition(16).persist()
print("finished normalizing")

finished normalizing


In [ ]:
full_df.head(1)

In [52]:
pca_input_cols = feature_cols
from dimensionality_reduction import fit_pca_model

pca_model, k_val = fit_pca_model(full_df.drop("label"), pca_input_cols, variance_target=0.95)

print(f"PCA model fitted with {k_val} components to capture 95% variance")

25/04/25 11:16:00 WARN DAGScheduler: Broadcasting large task binary with size 1442.3 KiB
25/04/25 11:16:00 WARN DAGScheduler: Broadcasting large task binary with size 1929.7 KiB
25/04/25 11:16:00 WARN DAGScheduler: Broadcasting large task binary with size 1929.7 KiB
25/04/25 11:16:00 WARN DAGScheduler: Broadcasting large task binary with size 1933.3 KiB
25/04/25 11:16:01 WARN DAGScheduler: Broadcasting large task binary with size 1934.4 KiB
25/04/25 11:16:01 WARN DAGScheduler: Broadcasting large task binary with size 1930.2 KiB
25/04/25 11:16:01 WARN DAGScheduler: Broadcasting large task binary with size 1931.9 KiB
25/04/25 11:16:02 WARN DAGScheduler: Broadcasting large task binary with size 1932.9 KiB
25/04/25 11:16:23 WARN DAGScheduler: Broadcasting large task binary with size 1442.3 KiB
25/04/25 11:16:23 WARN DAGScheduler: Broadcasting large task binary with size 1929.7 KiB
25/04/25 11:16:23 WARN DAGScheduler: Broadcasting large task binary with size 1929.7 KiB
25/04/25 11:16:23 WAR

PCA model fitted with 34 components to capture 95% variance


25/04/25 11:16:25 WARN DAGScheduler: Broadcasting large task binary with size 1932.9 KiB


In [53]:
pca_model.explainedVariance

DenseVector([0.5223, 0.084, 0.0641, 0.0422, 0.0347, 0.0328, 0.0162, 0.0145, 0.0117, 0.0106, 0.0095, 0.0091, 0.0082, 0.0082, 0.0079, 0.0074, 0.0061, 0.0058, 0.0054, 0.0051, 0.0049, 0.0044, 0.004, 0.0034, 0.0033, 0.0032, 0.0031, 0.0029, 0.0029, 0.0028, 0.0027, 0.0025, 0.0023, 0.0022])

In [54]:
pca_input_cols

['C3_Alpha_Power',
 'C3_Beta_Power',
 'C3_Delta_Power',
 'C3_Theta_Power',
 'C3_custom1_Power',
 'C4_Alpha_Power',
 'C4_Beta_Power',
 'C4_Delta_Power',
 'C4_Theta_Power',
 'C4_custom1_Power',
 'Cz_Alpha_Power',
 'Cz_Beta_Power',
 'Cz_Delta_Power',
 'Cz_Theta_Power',
 'Cz_custom1_Power',
 'F3_Alpha_Power',
 'F3_Beta_Power',
 'F3_Delta_Power',
 'F3_Theta_Power',
 'F3_custom1_Power',
 'F4_Alpha_Power',
 'F4_Beta_Power',
 'F4_Delta_Power',
 'F4_Theta_Power',
 'F4_custom1_Power',
 'F7_Alpha_Power',
 'F7_Beta_Power',
 'F7_Delta_Power',
 'F7_Theta_Power',
 'F7_custom1_Power',
 'F8_Alpha_Power',
 'F8_Beta_Power',
 'F8_Delta_Power',
 'F8_Theta_Power',
 'F8_custom1_Power',
 'Fp1_Alpha_Power',
 'Fp1_Beta_Power',
 'Fp1_Delta_Power',
 'Fp1_Theta_Power',
 'Fp1_custom1_Power',
 'Fp2_Alpha_Power',
 'Fp2_Beta_Power',
 'Fp2_Delta_Power',
 'Fp2_Theta_Power',
 'Fp2_custom1_Power',
 'Fz_Alpha_Power',
 'Fz_Beta_Power',
 'Fz_Delta_Power',
 'Fz_Theta_Power',
 'Fz_custom1_Power',
 'O1_Alpha_Power',
 'O1_Beta_P

In [55]:
pca_model.pc

DenseMatrix(145, 34, [-0.0964, -0.0996, 0.1117, -0.0978, -0.0863, -0.0962, -0.0988, 0.1114, ..., -0.0494, -0.0043, 0.004, 0.0041, -0.0378, -0.0044, 0.0041, 0.0288], 0)

In [56]:
import pandas as pd
import numpy as np

# Convert DenseMatrix to NumPy
pc_matrix = np.array(pca_model.pc.toArray())  # shape: (n_features, n_components)

# Create DataFrame of loadings
loadings_df = pd.DataFrame(pc_matrix, index=pca_input_cols, columns=[f"PC{val}" for val in range(1, k_val+1)])

# Get top 10 features for each component by absolute contribution
for pc in loadings_df.columns:
    print(f"\nTop features contributing to {pc}:")
    display(loadings_df[pc].abs().sort_values(ascending=False).head(10))



Top features contributing to PC1:


P4_Delta_Power    0.112594
Pz_Delta_Power    0.112325
P3_Delta_Power    0.112286
C3_Delta_Power    0.111667
C4_Delta_Power    0.111397
Fz_Delta_Power    0.111016
Cz_Delta_Power    0.110872
F4_Delta_Power    0.110508
T3_Delta_Power    0.110465
F3_Delta_Power    0.110444
Name: PC1, dtype: float64


Top features contributing to PC2:


Cz_TotalEnergy    0.182274
C3_TotalEnergy    0.181154
C4_TotalEnergy    0.180699
F3_TotalEnergy    0.177181
Fz_TotalEnergy    0.177119
F4_TotalEnergy    0.176684
Pz_TotalEnergy    0.175374
T3_TotalEnergy    0.175184
T4_TotalEnergy    0.174915
P4_TotalEnergy    0.174475
Name: PC2, dtype: float64


Top features contributing to PC3:


...

Pz_custom1_Power    0.226755
P4_custom1_Power    0.203114
F7_Beta_Power       0.202051
F4_Beta_Power       0.198495
F4_Delta_Power      0.194107
O2_Alpha_Power      0.177222
F8_Beta_Power       0.159258
C3_custom1_Power    0.157662
T3_custom1_Power    0.155428
F4_Theta_Power      0.155048
Name: PC32, dtype: float64


Top features contributing to PC33:


C4_custom1_Power    0.240288
O1_Delta_Power      0.219505
F7_custom1_Power    0.218797
O1_Alpha_Power      0.214499
T6_Beta_Power       0.204834
O1_custom1_Power    0.196857
C4_Alpha_Power      0.191517
O2_Beta_Power       0.174997
O1_Theta_Power      0.171146
O1_TotalEnergy      0.164509
Name: PC33, dtype: float64


Top features contributing to PC34:


F8_Beta_Power        0.244487
O2_custom1_Power     0.217160
O2_Alpha_Power       0.215200
T5_custom1_Power     0.195240
F8_Delta_Power       0.178712
T5_Alpha_Power       0.177694
O2_TotalEnergy       0.177424
Fp2_custom1_Power    0.158784
T6_custom1_Power     0.157349
T3_Beta_Power        0.156783
Name: PC34, dtype: float64

In [57]:
pca_input_cols

['C3_Alpha_Power',
 'C3_Beta_Power',
 'C3_Delta_Power',
 'C3_Theta_Power',
 'C3_custom1_Power',
 'C4_Alpha_Power',
 'C4_Beta_Power',
 'C4_Delta_Power',
 'C4_Theta_Power',
 'C4_custom1_Power',
 'Cz_Alpha_Power',
 'Cz_Beta_Power',
 'Cz_Delta_Power',
 'Cz_Theta_Power',
 'Cz_custom1_Power',
 'F3_Alpha_Power',
 'F3_Beta_Power',
 'F3_Delta_Power',
 'F3_Theta_Power',
 'F3_custom1_Power',
 'F4_Alpha_Power',
 'F4_Beta_Power',
 'F4_Delta_Power',
 'F4_Theta_Power',
 'F4_custom1_Power',
 'F7_Alpha_Power',
 'F7_Beta_Power',
 'F7_Delta_Power',
 'F7_Theta_Power',
 'F7_custom1_Power',
 'F8_Alpha_Power',
 'F8_Beta_Power',
 'F8_Delta_Power',
 'F8_Theta_Power',
 'F8_custom1_Power',
 'Fp1_Alpha_Power',
 'Fp1_Beta_Power',
 'Fp1_Delta_Power',
 'Fp1_Theta_Power',
 'Fp1_custom1_Power',
 'Fp2_Alpha_Power',
 'Fp2_Beta_Power',
 'Fp2_Delta_Power',
 'Fp2_Theta_Power',
 'Fp2_custom1_Power',
 'Fz_Alpha_Power',
 'Fz_Beta_Power',
 'Fz_Delta_Power',
 'Fz_Theta_Power',
 'Fz_custom1_Power',
 'O1_Alpha_Power',
 'O1_Beta_P

In [58]:
import importlib
try:
    importlib.reload(dimensionality_reduction)
except:
    pass
from dimensionality_reduction import apply_pca_model

full_df = apply_pca_model(full_df, pca_input_cols, pca_model, k_val)


In [59]:
import importlib
try:
    importlib.reload(dimensionality_reduction)
except:
    pass
    
from dimensionality_reduction import min_max_normalize, normalize_by_column, normalize_by_column_per_subject_wide

# train_df = normalize_by_column_per_subject_wide(train_df, feature_cols)
# test_df = normalize_by_column_per_subject_wide(test_df, feature_cols)


In [60]:
print("got here")

got here


# ML time

In [61]:
full_df.head(1)

25/04/25 11:18:09 WARN DAGScheduler: Broadcasting large task binary with size 1442.3 KiB
25/04/25 11:18:09 WARN DAGScheduler: Broadcasting large task binary with size 1976.6 KiB


[Row(SubjectID='sub-008', EpochID='ep-1016', label=1, features=DenseVector([6.7033, 1.8512, -0.6695, 1.9369, -1.4512, 1.1472, 0.3541, 0.3675, -1.2422, 0.4608, -0.131, 0.5302, -0.3766, -0.7561, -0.2705, -0.2397, -0.1048, 0.0011, -0.1653, 0.3432, 0.3639, -0.0908, 2.4207, -0.225, -0.2371, 0.0369, 0.3077, -0.251, 0.5335, -0.2506, 0.3853, 0.1466, -0.0227, 0.275]))]

In [62]:
full_df.columns

['SubjectID', 'EpochID', 'label', 'features']

In [63]:
import importlib
try:
    import dimensionality_reduction
    importlib.reload(dimensionality_reduction)
except:
    pass
importlib.reload(dimensionality_reduction)
from dimensionality_reduction import min_max_normalize_post_pca_by_subject

full_df = min_max_normalize_post_pca_by_subject(full_df)


25/04/25 11:21:47 WARN DAGScheduler: Broadcasting large task binary with size 1442.3 KiB
25/04/25 11:21:48 WARN DAGScheduler: Broadcasting large task binary with size 1974.7 KiB


In [64]:
full_df.head(1)

25/04/25 11:22:50 WARN DAGScheduler: Broadcasting large task binary with size 1442.3 KiB
25/04/25 11:22:52 WARN DAGScheduler: Broadcasting large task binary with size 1442.3 KiB
25/04/25 11:22:52 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
25/04/25 11:22:53 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB


[Row(SubjectID='sub-008', EpochID='ep-1016', label=1, features=DenseVector([0.9289, 0.1533, 0.5213, 0.4983, 0.197, 0.4374, 0.6963, 0.5187, 0.3857, 0.6028, 0.4149, 0.2251, 0.5973, 0.7719, 0.551, 0.1321, 0.4286, 0.4693, 0.5094, 0.4917, 0.6572, 0.2843, 0.7975, 0.427, 0.5214, 0.3768, 0.4432, 0.5377, 0.6167, 0.431, 0.7822, 0.5751, 0.6178, 0.5983]))]

In [65]:
full_df = full_df.toPandas()
# spark.stop()

25/04/25 11:23:35 WARN DAGScheduler: Broadcasting large task binary with size 1442.3 KiB
25/04/25 11:23:37 WARN DAGScheduler: Broadcasting large task binary with size 1442.3 KiB
25/04/25 11:23:37 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
25/04/25 11:23:39 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
                                                                                

In [66]:
full_df.head(1)

,SubjectID,EpochID,label,features
0,sub-008,ep-1016,1,"[0.9289157053211116, 0.15334788963250234, 0.52..."


In [ ]:
import numpy as np

# Convert Spark DenseVectors to regular 2D numpy arrays
X_train = np.array(train_pd["features"].tolist())
y_train = train_pd["label"].values

X_test = np.array(test_pd["features"].tolist())
y_test = test_pd["label"].values


In [ ]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)


In [ ]:
y_train

In [ ]:
X_train

In [ ]:
# How can we do standard scaler per subject ! !!!  ! ! ! !  !  !! ! !  ! 

In [ ]:
print("work")

In [ ]:
from sklearn.preprocessing import StandardScaler


X_train_scaled = X_train

X_test_scaled = X_test

# making sure min-maxed ! also might change results a little 

# scaler = StandardScaler()

# X_train_scaled = scaler.fit_transform(X_train)

# X_test_scaled = scaler.transform(X_test)

In [129]:
%%time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import numpy as np
import time

# # Step 1: Scale once
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# Step 2: Define hyperparameter grid
k_values = [1, 2, 3, 5, 7, 9, 11]
weights_list = ['uniform', 'distance']
metrics = ['euclidean', 'manhattan']
p_values = [1, 2]  # Only used if metric is 'minkowski'

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]
print("start")

# Step 4: Manual hyperparameter search
for k in k_values:
    for weight in weights_list:
        for metric in metrics:
            for p in p_values:

                if metric != 'minkowski' and p != 2:
                    continue  # p is irrelevant unless using 'minkowski'

                label = f"KNN k={k}, weight={weight}, metric={metric}, p={p}"
                model = KNeighborsClassifier(
                    n_neighbors=k,
                    weights=weight,
                    metric=metric if metric != 'minkowski' else 'minkowski',
                    p=p
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation scores
                scores = cross_val_score(model, X_train_scaled, y_train, cv=15, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Best fold evaluation
                skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_scaled[train_idx], X_train_scaled[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        val_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Validation Accuracy: {val_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        # Step 7: Evaluate on the held-out test set
                        #!! shouldn't we be testing oin the model that is trained on all the folds 1 by 1 (so multiple epochs) ?
                        model.fit(X_train_scaled, y_train)
                        y_test_pred = model.predict(X_test_scaled)
                        test_acc = accuracy_score(y_test, y_test_pred)
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_test, y_test_pred, target_names=target_names))

                        results.append((label, mean_acc, std_acc, train_acc, val_acc, test_acc))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 8: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, val_acc, test_acc in results:
    print(f"{label:<65} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Validation: {val_acc:.4f} | Test: {test_acc:.4f}")


start

=== Cross-Validation: KNN k=1, weight=uniform, metric=euclidean, p=2 ===
Mean Accuracy: 0.9996
Std Deviation: 0.0002
All Fold Scores: [0.9999 0.9995 0.9995 0.9997 0.9998 0.9998 0.9996 0.9996 0.9994 0.9996
 0.9996 0.9993 0.9996 0.9997 0.9995]

=== Best Fold Summary: KNN k=1, weight=uniform, metric=euclidean, p=2 ===
Train Accuracy: 1.0000
Validation Accuracy: 0.9998
              precision    recall  f1-score   support

     Control       1.00      1.00      1.00      5041
 Alzheimer's       1.00      1.00      1.00      6203

    accuracy                           1.00     11244
   macro avg       1.00      1.00      1.00     11244
weighted avg       1.00      1.00      1.00     11244

Test Accuracy: 0.4822
              precision    recall  f1-score   support

     Control       0.52      0.59      0.55      5552
 Alzheimer's       0.42      0.36      0.38      4624

    accuracy                           0.48     10176
   macro avg       0.47      0.47      0.47     10176
weig

KeyboardInterrupt: 

In [71]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

def subject_knn_recall(df, max_subjects=10, test_size=0.3):
    # Make sure features are proper numpy arrays
    df = df.copy()
    df['features'] = df['features'].apply(lambda x: np.array(x))
    
    # Extract feature matrix
    X = np.stack(df['features'].values)
    y = df['SubjectID'].values
    df['SubjectID'] = y  # just to be sure they're strings

    subjects = df['SubjectID'].unique()
    np.random.shuffle(subjects)

    results = []

    for n_subjects in range(2, min(max_subjects + 1, len(subjects) + 1)):
        selected = subjects[:n_subjects]
        df_sel = df[df['SubjectID'].isin(selected)]

        # Split train/test *per subject*
        train_list, val_list = [], []
        for subj in selected:
            subj_df = df_sel[df_sel['SubjectID'] == subj]
            train, val = train_test_split(subj_df, test_size=test_size, random_state=42)
            train_list.append(train)
            val_list.append(val)

        train_df = pd.concat(train_list)
        val_df = pd.concat(val_list)

        X_train = np.stack(train_df['features'].values)
        y_train = train_df['SubjectID'].values

        X_val = np.stack(val_df['features'].values)
        y_val = val_df['SubjectID'].values

        clf = KNeighborsClassifier(n_neighbors=1, weights='uniform', metric='minkowski', p=2)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_val)

        recall = recall_score(y_val, y_pred, average='macro')  # You could also try 'micro'
        results.append((n_subjects, recall))
        print(f"{n_subjects} subjects — Recall (macro): {recall:.4f}")

    return pd.DataFrame(results, columns=["Num_Subjects", "Recall"])

# Example run:
recall_df = subject_knn_recall(full_df, max_subjects=65, test_size=0.40)


2 subjects — Recall (macro): 1.0000
3 subjects — Recall (macro): 1.0000
4 subjects — Recall (macro): 0.9998
5 subjects — Recall (macro): 0.9996
6 subjects — Recall (macro): 0.9996
7 subjects — Recall (macro): 0.9995
8 subjects — Recall (macro): 0.9994
9 subjects — Recall (macro): 0.9995
10 subjects — Recall (macro): 0.9996
11 subjects — Recall (macro): 0.9995
12 subjects — Recall (macro): 0.9996
13 subjects — Recall (macro): 0.9996
14 subjects — Recall (macro): 0.9992
15 subjects — Recall (macro): 0.9992
16 subjects — Recall (macro): 0.9991
17 subjects — Recall (macro): 0.9990
18 subjects — Recall (macro): 0.9990
19 subjects — Recall (macro): 0.9989
20 subjects — Recall (macro): 0.9989
21 subjects — Recall (macro): 0.9989
22 subjects — Recall (macro): 0.9985
23 subjects — Recall (macro): 0.9984
24 subjects — Recall (macro): 0.9982
25 subjects — Recall (macro): 0.9982
26 subjects — Recall (macro): 0.9978
27 subjects — Recall (macro): 0.9979
28 subjects — Recall (macro): 0.9978
29 subjec

In [77]:
def finger(test_size):
    print(f"test_size: {test_size}")
    recall_df = subject_knn_recall(full_df, max_subjects=65, test_size=test_size)

# finger(0.40)
finger(0.50)
finger(0.60)
finger(0.70)
finger(0.80)
finger(0.90)
finger(0.95)

test_size: 0.5
2 subjects — Recall (macro): 1.0000
3 subjects — Recall (macro): 0.9995
4 subjects — Recall (macro): 0.9992
5 subjects — Recall (macro): 0.9994
6 subjects — Recall (macro): 0.9983
7 subjects — Recall (macro): 0.9975
8 subjects — Recall (macro): 0.9978
9 subjects — Recall (macro): 0.9978
10 subjects — Recall (macro): 0.9979
11 subjects — Recall (macro): 0.9981
12 subjects — Recall (macro): 0.9980
13 subjects — Recall (macro): 0.9975
14 subjects — Recall (macro): 0.9976
15 subjects — Recall (macro): 0.9975
16 subjects — Recall (macro): 0.9976
17 subjects — Recall (macro): 0.9972
18 subjects — Recall (macro): 0.9961
19 subjects — Recall (macro): 0.9959
20 subjects — Recall (macro): 0.9959
21 subjects — Recall (macro): 0.9958
22 subjects — Recall (macro): 0.9960
23 subjects — Recall (macro): 0.9950
24 subjects — Recall (macro): 0.9951
25 subjects — Recall (macro): 0.9949
26 subjects — Recall (macro): 0.9946
27 subjects — Recall (macro): 0.9943
28 subjects — Recall (macro): 0

In [130]:
%%time
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import time

# Step 1: Scale data once (IMPORTANT: transform X_test with same scaler)
scaler = MinMaxScaler(feature_range=(-1, 1))
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 2: Hyperparameters
layer_configs = [(256, 128, 64), (128, 64, 16)]
activations = ['relu']
alphas = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 5e-1, 1.0]
early_stopping_options = [True, False]
max_iter = 30000

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Brute-force hyperparameter loop
for layers in layer_configs:
    for activation in activations:
        for alpha in alphas:
            for early_stopping in early_stopping_options:

                label = f"MLP {layers}, act={activation}, alpha={alpha}, early_stop={early_stopping}, max_iter={max_iter}"
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation=activation,
                    alpha=alpha,
                    early_stopping=early_stopping,
                    max_iter=max_iter,
                    random_state=42,
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation
                scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Evaluate on best validation fold
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_scaled[train_idx], X_train_scaled[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        val_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Validation Accuracy: {val_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        break

                # Step 7: Evaluate on held-out test set
                model.fit(X_train_scaled, y_train)
                y_test_pred = model.predict(X_test_scaled)
                test_acc = accuracy_score(y_test, y_test_pred)
                print(f"\n=== Final Test Set Evaluation: {label} ===")
                print(f"Test Accuracy: {test_acc:.4f}")
                print(classification_report(y_test, y_test_pred, target_names=target_names))

                results.append((label, mean_acc, std_acc, train_acc, val_acc, test_acc))

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 8: Print final summary
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, val_acc, test_acc in results:
    print(f"{label:<75} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Validation: {val_acc:.4f} | Test: {test_acc:.4f}")



=== Cross-Validation: MLP (256, 128, 64), act=relu, alpha=1e-06, early_stop=True, max_iter=30000 ===
Mean Accuracy: 0.9913
Std Deviation: 0.0006
All Fold Scores: [0.9921 0.9918 0.9912 0.9902 0.9913]

=== Best Fold Summary: MLP (256, 128, 64), act=relu, alpha=1e-06, early_stop=True, max_iter=30000 ===
Train Accuracy: 0.9991
Validation Accuracy: 0.9921
              precision    recall  f1-score   support

     Control       0.99      0.99      0.99     15123
 Alzheimer's       0.99      0.99      0.99     18608

    accuracy                           0.99     33731
   macro avg       0.99      0.99      0.99     33731
weighted avg       0.99      0.99      0.99     33731


=== Final Test Set Evaluation: MLP (256, 128, 64), act=relu, alpha=1e-06, early_stop=True, max_iter=30000 ===
Test Accuracy: 0.5503
              precision    recall  f1-score   support

     Control       0.60      0.53      0.56      5552
 Alzheimer's       0.50      0.57      0.54      4624

    accuracy          

/Users/admin/neuro-venv/lib/python3.9/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/Users/admin/neuro-venv/lib/python3.9/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/Users/admin/neuro-venv/lib/python3.9/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


KeyboardInterrupt: 

In [131]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np

# Define models
models = {
    # "KNN-tuned": KNeighborsClassifier(
    #     n_neighbors=3,
    #     weights='distance',
    #     metric='euclidean',
    #     p=1
    # ),
    # "KNN": KNeighborsClassifier(),
    # "NeuralNet": MLPClassifier(hidden_layer_sizes=(100,), max_iter=10000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(
        max_depth=8,
        max_features=None,
        min_samples_leaf=10,
        min_samples_split=5,
        random_state=42
    ),
    "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "BaggedSVM": make_pipeline(
        BaggingClassifier(
            estimator=SVC(kernel='linear', probability=False),
            n_estimators=10,
            max_samples=0.1,
            n_jobs=3,
            bootstrap=False,
            random_state=42
        )
    ),
    "SVM": make_pipeline(
        SVC(kernel='linear', probability=True)
    )
}

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # Refit on best 14/15 folds and evaluate on the 1/15
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            
            model.fit(X_tr, y_tr)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)

            val_acc = accuracy_score(y_te, y_pred_test)
            train_acc = accuracy_score(y_tr, y_pred_train)

            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Train Accuracy: {train_acc:.4f}")
            print(f"Validation Accuracy: {val_acc:.4f}")
            print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
            break

    # Final evaluation on the held-out test set
    model.fit(X_train, y_train)
    y_test_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)
    print(f"\n=== Test Set Evaluation: {name} ===")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(classification_report(y_test, y_test_pred, target_names=["Control", "Alzheimer's"]))



=== Cross-Validation: DecisionTree ===
Mean Accuracy: 0.8532
Standard Deviation: 0.0057
All Fold Scores: [0.8587 0.8495 0.8602 0.8552 0.8479 0.8416 0.8492 0.8636 0.854  0.8495
 0.8513 0.8532 0.8605 0.8562 0.8478]

=== Best Fold Summary: DecisionTree ===
Train Accuracy: 0.8638
Validation Accuracy: 0.8588
              precision    recall  f1-score   support

     Control       0.85      0.84      0.84      5040
 Alzheimer's       0.87      0.88      0.87      6203

    accuracy                           0.86     11243
   macro avg       0.86      0.86      0.86     11243
weighted avg       0.86      0.86      0.86     11243


=== Test Set Evaluation: DecisionTree ===
Test Accuracy: 0.6730
              precision    recall  f1-score   support

     Control       0.67      0.78      0.72      5552
 Alzheimer's       0.68      0.54      0.60      4624

    accuracy                           0.67     10176
   macro avg       0.67      0.66      0.66     10176
weighted avg       0.67      0

/Users/admin/neuro-venv/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Mean Accuracy: 0.7600
Standard Deviation: 0.0139
All Fold Scores: [0.7524 0.7582 0.7828 0.7649 0.743  0.7436 0.7562 0.7812 0.7624 0.746
 0.7534 0.7616 0.7893 0.7572 0.7478]

=== Best Fold Summary: BaggedSVM ===
Train Accuracy: 0.7630
Validation Accuracy: 0.7561
              precision    recall  f1-score   support

     Control       0.75      0.68      0.71      5041
 Alzheimer's       0.76      0.82      0.79      6202

    accuracy                           0.76     11243
   macro avg       0.76      0.75      0.75     11243
weighted avg       0.76      0.76      0.75     11243


=== Test Set Evaluation: BaggedSVM ===
Test Accuracy: 0.7130
              precision    recall  f1-score   support

     Control       0.69      0.86      0.77      5552
 Alzheimer's       0.77      0.53      0.63      4624

    accuracy                           0.71     10176
   macro avg       0.73      0.70      0.70     10176
weighted avg       0.72      0.71      0.70     10176


=== Cross-Validation:

KeyboardInterrupt: 

In [134]:
%%time
from sklearn.ensemble import BaggingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import time

# Step 1: Scale data for SVMs
scaler = StandardScaler()
X_train_svm = scaler.fit_transform(X_train)
X_test_svm = scaler.transform(X_test)

# Step 2: Define hyperparameter grids
C_values = [0.01, 0.1, 1, 10]
n_estimators_list = [5, 10, 20]
max_samples_list = [0.1, 0.5, 1.0]
bootstrap_options = [False, True]

# Step 3: Track results
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Hyperparameter tuning
for C in C_values:
    for n_est in n_estimators_list:
        for max_samp in max_samples_list:
            for bootstrap in bootstrap_options:
                
                label = f"BaggedSVM C={C}, est={n_est}, max_samples={max_samp}, bootstrap={bootstrap}"
                model = make_pipeline(
                    BaggingClassifier(
                        estimator=SVC(C=C, kernel='linear', probability=False),
                        n_estimators=n_est,
                        max_samples=max_samp,
                        bootstrap=bootstrap,
                        n_jobs=3,
                        random_state=42
                    )
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                scores = cross_val_score(model, X_train_svm, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Best fold deep dive
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, val_idx) in enumerate(skf.split(X_train_svm, y_train)):
                    if i == best_fold_index:
                        X_tr, X_val = X_train_svm[train_idx], X_train_svm[val_idx]
                        y_tr, y_val = y_train[train_idx], y_train[val_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_val = model.predict(X_val)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        val_acc = accuracy_score(y_val, y_pred_val)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Validation Accuracy: {val_acc:.4f}")
                        print(classification_report(y_val, y_pred_val, target_names=target_names))

                        break

                # Final test set evaluation
                model.fit(X_train_svm, y_train)
                y_test_pred = model.predict(X_test_svm)
                test_acc = accuracy_score(y_test, y_test_pred)

                print(f"\n=== Test Set Evaluation: {label} ===")
                print(f"Test Accuracy: {test_acc:.4f}")
                print(classification_report(y_test, y_test_pred, target_names=target_names))

                results.append((label, mean_acc, std_acc, train_acc, val_acc, test_acc))

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 5: Print top models
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, val_acc, test_acc in results:
    print(f"{label:<90} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f}")



=== Cross-Validation: BaggedSVM C=0.01, est=5, max_samples=0.1, bootstrap=False ===
Mean Accuracy: 0.7603
Std Deviation: 0.0069
All Fold Scores: [0.7661 0.7495 0.7672 0.7547 0.764 ]

=== Best Fold Summary: BaggedSVM C=0.01, est=5, max_samples=0.1, bootstrap=False ===
Train Accuracy: 0.7618
Validation Accuracy: 0.7632
              precision    recall  f1-score   support

     Control       0.77      0.67      0.72     15122
 Alzheimer's       0.76      0.84      0.80     18608

    accuracy                           0.76     33730
   macro avg       0.76      0.75      0.76     33730
weighted avg       0.76      0.76      0.76     33730


=== Test Set Evaluation: BaggedSVM C=0.01, est=5, max_samples=0.1, bootstrap=False ===
Test Accuracy: 0.7080
              precision    recall  f1-score   support

     Control       0.69      0.85      0.76      5552
 Alzheimer's       0.75      0.54      0.63      4624

    accuracy                           0.71     10176
   macro avg       0.72  

KeyboardInterrupt: 

In [ ]:
%%time
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import time

# Step 1: No need to scale for trees
X_train_tree = X_train
X_test_tree = X_test

# Step 2: More complex hyperparameter grid
max_depths = [10, 15, 20, None]  # None = fully grow the tree
min_samples_splits = [2, 3, 5]   # Smaller split thresholds
min_samples_leafs = [1, 2]       # Smaller leaves allowed

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Manual hyperparameter search
for depth in max_depths:
    for min_split in min_samples_splits:
        for min_leaf in min_samples_leafs:

            label = f"DecisionTree max_depth={depth}, min_split={min_split}, min_leaf={min_leaf}"
            model = DecisionTreeClassifier(
                max_depth=depth,
                min_samples_split=min_split,
                min_samples_leaf=min_leaf,
                max_features=None,  # use all features
                ccp_alpha=0.0,      # disable pruning
                random_state=42
            )

            print(f"\n=== Cross-Validation: {label} ===")
            start = time.time()

            # Step 5: Cross-validation scores
            scores = cross_val_score(model, X_train_tree, y_train, cv=5, scoring='accuracy', n_jobs=3)
            mean_acc, std_acc = scores.mean(), scores.std()
            print(f"Mean Accuracy: {mean_acc:.4f}")
            print(f"Std Deviation: {std_acc:.4f}")
            print(f"All Fold Scores: {np.round(scores, 4)}")

            # Step 6: Best fold evaluation
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            best_fold_index = np.argmax(scores)
            for i, (train_idx, test_idx) in enumerate(skf.split(X_train_tree, y_train)):
                if i == best_fold_index:
                    X_tr, X_te = X_train_tree[train_idx], X_train_tree[test_idx]
                    y_tr, y_te = y_train[train_idx], y_train[test_idx]

                    model.fit(X_tr, y_tr)
                    y_pred_test = model.predict(X_te)
                    y_pred_train = model.predict(X_tr)

                    train_acc = accuracy_score(y_tr, y_pred_train)
                    test_acc = accuracy_score(y_te, y_pred_test)

                    print(f"\n=== Best Fold Summary: {label} ===")
                    print(f"Train Accuracy: {train_acc:.4f}")
                    print(f"Test Accuracy: {test_acc:.4f}")
                    print(classification_report(y_te, y_pred_test, target_names=target_names))

                    results.append((label, mean_acc, std_acc, train_acc, test_acc))
                    break

            print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 7: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<75} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")


In [ ]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np


# Define models
models = { # NEED TO GO THORUGH AND MAKE SURE DEFAULTS ARE OK, SHOULD NOT HAVE HAD STANDARD SCALER FOR SVM'S
    # "KNN-tuned": KNeighborsClassifier(
    # n_neighbors=3,
    # weights='distance',
    # metric='euclidean',
    # p=1
    # ),
    # "KNN": KNeighborsClassifier(),
    # "NeuralNet": MLPClassifier(hidden_layer_sizes=(100,), max_iter=10000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(
        max_depth=8,
        max_features=None,
        min_samples_leaf=10,
        min_samples_split=5,
        random_state=42
    ),
    "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42),
        # Faster ensemble of SVMs
    "BaggedSVM": make_pipeline(
        # StandardScaler(),
        BaggingClassifier(
            estimator=SVC(kernel='linear', probability=False),
            n_estimators=10,
            max_samples=0.1,
            n_jobs=3,
            bootstrap=False,
            random_state=42
        )
    ),

    # Classic SVM (can be slow on big data)
    "SVM": make_pipeline(
        # StandardScaler(), 
        SVC(kernel='linear',
        probability=True)
    )
}

from sklearn.model_selection import StratifiedKFold

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # Refit on best 14/15 folds and evaluate on the 1/15
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            
            model.fit(X_tr, y_tr)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)

            test_acc = accuracy_score(y_te, y_pred_test)
            train_acc = accuracy_score(y_tr, y_pred_train)

            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Train Accuracy: {train_acc:.4f}")
            print(f"Test Accuracy: {test_acc:.4f}")
            print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
            break


# ML experiments and tuning

Notes and concerns:
 Are we folding correclty, it looks like we are retraining the best fold, ??? super weird
 Also seems like we need to do more iterations for SVM ... we can get some more out of it 

 Since hte neural nets that havve best aucuracy almost have 1000% in training, should try and do regurlization to try and 'even' out the accuracy between the two

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
import numpy as np
import time


import warnings
from sklearn.exceptions import ConvergenceWarning

# Suppress ConvergenceWarnings only
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Set up test space
kernels = ['linear', 'rbf', 'sigmoid']
Cs = [0.1, 1, 10]
gammas = ['scale', 0.01]  # for rbf/sigmoid
n_estimators = 10
max_iter = 10000

# Replace with your target class names
target_names = ["Control", "Alzheimer's"]

# Loop over kernel/C/gamma
for kernel in kernels:
    for C in Cs:
        if kernel == 'linear':
            label = f"BaggedSVM - linear, C={C}"
            base_model = SVC(kernel=kernel, C=C, probability=False, max_iter=max_iter)
            model = make_pipeline(
                BaggingClassifier(
                    estimator=base_model,
                    n_estimators=n_estimators,
                    max_samples=0.1,
                    n_jobs=3,
                    bootstrap=False,
                    random_state=42
                )
            )
            print(f"\n=== Cross-Validation: {label} ===")
            start = time.time()

            scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
            print(f"Mean Accuracy: {scores.mean():.4f}")
            print(f"Standard Deviation: {scores.std():.4f}")
            print(f"All Fold Scores: {np.round(scores, 4)}")

            best_fold_index = np.argmax(scores)
            skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
            for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
                if i == best_fold_index:
                    X_tr, X_te = X_train[train_index], X_train[test_index]
                    y_tr, y_te = y_train[train_index], y_train[test_index]

                    model.fit(X_tr, y_tr)
                    y_pred_test = model.predict(X_te)
                    y_pred_train = model.predict(X_tr)

                    test_acc = accuracy_score(y_te, y_pred_test)
                    train_acc = accuracy_score(y_tr, y_pred_train)

                    print(f"\n=== Best Fold Summary: {label} ===")
                    print(f"Train Accuracy: {train_acc:.4f}")
                    print(f"Test Accuracy: {test_acc:.4f}")
                    print(classification_report(y_te, y_pred_test, target_names=target_names))
                    break

            print(f"⏱️ Duration: {time.time() - start:.1f} sec")

        elif kernel in ['rbf', 'sigmoid']:
            for gamma in gammas:
                label = f"BaggedSVM - {kernel}, C={C}, gamma={gamma}"
                base_model = SVC(kernel=kernel, C=C, gamma=gamma, probability=False, max_iter=max_iter)
                model = make_pipeline(
                    BaggingClassifier(
                        estimator=base_model,
                        n_estimators=n_estimators,
                        max_samples=0.1,
                        n_jobs=3,
                        bootstrap=False,
                        random_state=42
                    )
                )
                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
                print(f"Mean Accuracy: {scores.mean():.4f}")
                print(f"Standard Deviation: {scores.std():.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                best_fold_index = np.argmax(scores)
                skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
                for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train[train_index], X_train[test_index]
                        y_tr, y_te = y_train[train_index], y_train[test_index]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        test_acc = accuracy_score(y_te, y_pred_test)
                        train_acc = accuracy_score(y_tr, y_pred_train)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f} sec")


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import time

# Step 1: Scale data once
scaler = MinMaxScaler(feature_range=(-1, 1))  # You can use (0, 1) if you prefer
X_train_scaled = scaler.fit_transform(X_train)

# Step 2: Define hyperparameter grids
layer_configs = [(100,), (128,), (128, 64), (256, 128, 64)]
activations = ['relu', 'tanh']
alphas = [1e-4, 1e-3, 1e-2]
early_stopping_options = [True, False]

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Brute-force loop over hyperparameter combinations
for layers in layer_configs:
    for activation in activations:
        for alpha in alphas:
            for early_stopping in early_stopping_options:

                label = f"MLP {layers}, act={activation}, alpha={alpha}, early_stop={early_stopping}"
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation=activation,
                    alpha=alpha,
                    early_stopping=early_stopping,
                    max_iter=10000,
                    random_state=42,
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation accuracy scores
                scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Find best fold for detailed summary
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_scaled[train_idx], X_train_scaled[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        test_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        results.append((label, mean_acc, std_acc, train_acc, test_acc))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 7: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<65} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")


In [ ]:
%%time
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import time

# Step 1: Scale features if needed (optional for GB, but useful with continuous variables)
X_train_boost = X_train_scaled  # assuming you already have this
X_test_boost = X_test_scaled

# Step 2: Define hyperparameter grid for more complex boosting
n_estimators_list = [100, 200]
learning_rates = [0.05, 0.1]
max_depths = [3, 5, 7]
min_samples_leafs = [1, 3]

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Manual hyperparameter search
for n_est in n_estimators_list:
    for lr in learning_rates:
        for depth in max_depths:
            for min_leaf in min_samples_leafs:

                label = f"GradBoost n={n_est}, lr={lr}, depth={depth}, min_leaf={min_leaf}"
                model = GradientBoostingClassifier(
                    n_estimators=n_est,
                    learning_rate=lr,
                    max_depth=depth,
                    min_samples_leaf=min_leaf,
                    random_state=42
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation scores
                scores = cross_val_score(model, X_train_boost, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Best fold evaluation
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_boost, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_boost[train_idx], X_train_boost[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        test_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        results.append((label, mean_acc, std_acc, train_acc, test_acc))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 7: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<75} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")


In [ ]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np

import os

# Define models
models = { # NEED TO GO THORUGH AND MAKE SURE DEFAULTS ARE OK, SHOULD NOT HAVE HAD STANDARD SCALER FOR SVM'S
    # "KNN": KNeighborsClassifier(),
    # "NeuralNet": MLPClassifier(hidden_layer_sizes=(100,), max_iter=10000, random_state=42),
    # "DecisionTree": DecisionTreeClassifier(
    #     max_depth=8,
    #     max_features=None,
    #     min_samples_leaf=10,
    #     min_samples_split=5,
    #     random_state=42
    # ),
    # "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42),
        # Faster ensemble of SVMs
    # "BaggedSVM": make_pipeline(
    #     StandardScaler(),
    #     BaggingClassifier(
    #         estimator=SVC(kernel='linear', probability=False),
    #         n_estimators=10,
    #         max_samples=0.1,
    #         n_jobs=3,
    #         bootstrap=False,
    #         random_state=42
    #     )
    # ),
    # Classic SVM (can be slow on big data)
    "BaggedSVM": make_pipeline(
        # StandardScaler(),
        BaggingClassifier(
            estimator=SVC(kernel='linear', probability=False),
            n_estimators=10,
            max_samples=0.1,
            n_jobs=3,
            bootstrap=False,
            random_state=42
        )
    ),
}

from sklearn.model_selection import StratifiedKFold

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # Refit on best 14/15 folds and evaluate on the 1/15
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            
            model.fit(X_tr, y_tr)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)

            test_acc = accuracy_score(y_te, y_pred_test)
            train_acc = accuracy_score(y_tr, y_pred_train)

            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Train Accuracy: {train_acc:.4f}")
            print(f"Test Accuracy: {test_acc:.4f}")
            print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
            break

os.system('say "SVM is done!"')

# ML Testing/tuning # best for nets, tanh 200 hidden layers

In [ ]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np

# Define models
models = {
    # "KNN": KNeighborsClassifier(),
    # "SVM": make_pipeline(StandardScaler(), SVC(probability=True)),
    "NeuralNet": MLPClassifier(hidden_layer_sizes=(200,), activation='tanh', max_iter=10000, random_state=42)
    # "DecisionTree": DecisionTreeClassifier(
    #     max_depth=8,
    #     max_features=None,
    #     min_samples_leaf=10,
    #     min_samples_split=5,
    #     random_state=42
    # ),
    # "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

from sklearn.model_selection import StratifiedKFold

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # Refit on best 14/15 folds and evaluate on the 1/15
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            
            model.fit(X_tr, y_tr)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)

            test_acc = accuracy_score(y_te, y_pred_test)
            train_acc = accuracy_score(y_tr, y_pred_train)

            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Train Accuracy: {train_acc:.4f}")
            print(f"Test Accuracy: {test_acc:.4f}")
            print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
            break


In [ ]:
model = DecisionTreeClassifier(
    max_depth=8,
    max_features=None,
    min_samples_leaf=10,
    min_samples_split=5,
    random_state=42
)

print(f"\n=== Cross-Validation: Regularized DecisionTree ===")

# Cross-validation scores
scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)

print(f"Mean Accuracy: {scores.mean():.4f}")
print(f"Standard Deviation: {scores.std():.4f}")
print(f"All Fold Scores: {np.round(scores, 4)}")

# Find the best-performing fold
best_fold_index = np.argmax(scores)

# Evaluate best fold manually
skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
    if i == best_fold_index:
        X_tr, X_te = X_train[train_index], X_train[test_index]
        y_tr, y_te = y_train[train_index], y_train[test_index]

        model.fit(X_tr, y_tr)

        y_pred_train = model.predict(X_tr)
        y_pred_test = model.predict(X_te)

        train_acc = accuracy_score(y_tr, y_pred_train)
        test_acc = accuracy_score(y_te, y_pred_test)

        print(f"\n=== Best Fold Summary: Regularized DecisionTree ===")
        print(f"Train Accuracy: {train_acc:.4f}")
        print(f"Test Accuracy: {test_acc:.4f}")
        print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
        break


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

param_grid = {
    'max_depth': [4, 6, 8],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [5, 10, 20],
    'max_features': ['sqrt', 'log2', None]
}

tree = DecisionTreeClassifier(random_state=42)
grid_search = GridSearchCV(tree, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)